# Indian Food Delivery App Sentiment Intelligence System
## A Multi-Source NLP Analysis of Zomato, Swiggy & Blinkit Reviews

---

**Objective:** Analyze real customer reviews from Google Play Store 
to extract sentiment and business insights using NLP and Machine Learning.

**Apps Covered:** Zomato | Swiggy | Blinkit

**Run notebooks in order:**
1. 01_Data_Collection
2. 02_EDA
3. 03_Deep_Learning
4. 04_Business_Insights

In [1]:
!pip install scikit-learn nltk

In [2]:
!pip install textblob 

In [3]:
!pip install vaderSentiment

In [4]:
!pip install google-play-scraper

In [5]:
!pip install praw wordcloud

## 1. Importing Libraries

Importing all required libraries for data collection, 
text processing, and sentiment analysis.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wordcloud as WordCloud

In [7]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [8]:
from google_play_scraper import reviews, Sort
import warnings
warnings.filterwarnings('ignore')

print("All Libraries Imported Successfully!")

All Libraries Imported Successfully!


In [9]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

print("NLTK Data Downloaded Successfully!")

NLTK Data Downloaded Successfully!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 2. Data Collection

Collecting real user reviews directly from Google Play Store
for Zomato, Swiggy and Blinkit using google-play-scraper library.

**Why this approach?**
- Real data ensures 100% originality
- India-specific reviews for local business insights
- Multi-source data for better analysis

In [11]:
print("Data collection started... Please wait!")
zomato_reviews, _ = reviews(
    'com.application.zomato',
    lang='en',
    country='in',
    sort=Sort.NEWEST,
    count=500
)
print(f" Zomato: {len(zomato_reviews)} reviews fetched!")

swiggy_reviews, _ = reviews(
    'in.swiggy.android',
    lang='en',
    country='in',
    sort=Sort.NEWEST,
    count=500
)
print(f" Swiggy: {len(swiggy_reviews)} reviews fetched!")

blinkit_reviews, _ = reviews(
    'com.grofers.customerapp',
    lang='en',
    country='in',
    sort=Sort.NEWEST,
    count=500
)
print(f" Blinkit: {len(blinkit_reviews)} reviews fetched!")

Data collection started... Please wait!
 Zomato: 500 reviews fetched!
 Swiggy: 500 reviews fetched!
 Blinkit: 500 reviews fetched!


In [15]:
df_zomato = pd.DataFrame(zomato_reviews)[['userName', 'content', 'score', 'at']]
df_zomato['app'] = 'Zomato'

In [16]:
df_swiggy = pd.DataFrame(swiggy_reviews)[['userName', 'content', 'score', 'at']]
df_swiggy['app'] = 'Swiggy'

In [17]:
df_blinkit = pd.DataFrame(blinkit_reviews)[['userName', 'content', 'score', 'at']]
df_blinkit['app'] = 'Blinkit'

In [18]:
#Combinning the Data
df = pd.concat([df_zomato, df_swiggy, df_blinkit], ignore_index=True)

In [19]:
df.columns = ['username', 'review', 'rating', 'date', 'app']

print(f" Total Reviews: {len(df)}")
print(f"\n Shape: {df.shape}")
print(f"\n First 5 rows:")
df.head()

 Total Reviews: 1500

 Shape: (1500, 5)

 First 5 rows:


,username,review,rating,date,app
0,Sanjoy Manna,vijj,1,2026-03-12 02:48:43,Zomato
1,Sherin.S sherin,very good,5,2026-03-12 02:47:09,Zomato
2,Ningreithem Moilee,Plenty of options to choose anytime anywhere.,5,2026-03-12 02:45:45,Zomato
3,Md tunna,good,5,2026-03-12 02:30:00,Zomato
4,Naresh Kumar,ok,5,2026-03-12 02:25:01,Zomato


In [20]:
df.to_csv(r'C:\Users\Dell\Data_Science_Project\05_NLP_Sentiments_Project\Data\Raw\raw_reviews.csv', index=False)

print(" Raw data saved successfully!")
print(f" Location: data/raw/raw_reviews.csv")

 Raw data saved successfully!
 Location: data/raw/raw_reviews.csv


## 3. Raw Data Overview

Basic overview of collected dataset to understand 
structure, check data types and identify missing values.

In [22]:
print("=" * 50)
print(" DATASET OVERVIEW")
print("=" * 50)

print(f"\n Total Reviews: {len(df)}")
print(f" Columns: {list(df.columns)}")
print(f" Shape: {df.shape}")

print("\n Reviews per App:")
print(df['app'].value_counts())

print("\n Rating Distribution:")
print(df['rating'].value_counts().sort_index())

print("\n Missing Values:")
print(df.isnull().sum())

print("\n Data Types:")
print(df.dtypes)

 DATASET OVERVIEW

 Total Reviews: 1500
 Columns: ['username', 'review', 'rating', 'date', 'app']
 Shape: (1500, 5)

 Reviews per App:
app
Zomato     500
Swiggy     500
Blinkit    500
Name: count, dtype: int64

 Rating Distribution:
rating
1    322
2     30
3     67
4    162
5    919
Name: count, dtype: int64

 Missing Values:
username    0
review      0
rating      0
date        0
app         0
dtype: int64

 Data Types:
username               str
review                 str
rating               int64
date        datetime64[us]
app                    str
dtype: object


## 4. Data Preprocessing & Cleaning

Cleaning raw text data by removing URLs, special characters,
numbers and converting to lowercase for standardization.

In [23]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

In [24]:
df['cleaned_review'] = df['review'].apply(clean_text)
print("Original vs Cleaned Review:")
print("-" * 50)
for i in range(3):
    print(f"Original : {df['review'][i]}")
    print(f"Cleaned  : {df['cleaned_review'][i]}")
    print("-" * 50)

Original vs Cleaned Review:
--------------------------------------------------
Original : vijj
Cleaned  : vijj
--------------------------------------------------
Original : very good
Cleaned  : very good
--------------------------------------------------
Original : Plenty of options to choose anytime anywhere.
Cleaned  : plenty of options to choose anytime anywhere
--------------------------------------------------


## 5. Tokenization & Lemmatization

Breaking text into individual tokens, removing stopwords 
and converting words to their root form using lemmatization.

**Example:**
- "ordering food was amazing!" → "order food amazing"
- Stopwords removed: "was", "the", "is"
- Lemmatization: "ordering" → "order"

In [25]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [27]:
def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) 
              for word in tokens 
              if word not in stop_words 
              and len(word) > 2]
    return ' '.join(tokens)

In [28]:
df['processed_review'] = df['cleaned_review'].apply(preprocess_text)

In [29]:
print("Cleaned vs Processed Review:")
print("-" * 50)
for i in range(3):
    print(f"Cleaned   : {df['cleaned_review'][i]}")
    print(f"Processed : {df['processed_review'][i]}")
    print("-" * 50)

Cleaned vs Processed Review:
--------------------------------------------------
Cleaned   : vijj
Processed : vijj
--------------------------------------------------
Cleaned   : very good
Processed : good
--------------------------------------------------
Cleaned   : plenty of options to choose anytime anywhere
Processed : plenty option choose anytime anywhere
--------------------------------------------------


## 6. Sentiment Labeling

Applying two approaches for sentiment labeling:
- **VADER:** NLP based sentiment from review text
- **Rating Based:** Rule based sentiment from star ratings

Comparing both helps identify gap between written 
sentiment and rating behavior.

In [30]:
analyzer = SentimentIntensityAnalyzer()

In [31]:
def get_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

In [32]:
df['sentiment'] = df['processed_review'].apply(get_sentiment)

In [34]:
def rating_to_sentiment(rating):
    if rating >= 4:
        return 'Positive'
    elif rating == 3:
        return 'Neutral'
    else:
        return 'Negative'
df['rating_sentiment'] = df['rating'].apply(rating_to_sentiment)

In [35]:
print("Sentiment Distribution (VADER):")
print("-" * 50)
print(df['sentiment'].value_counts())
print("\nSentiment Distribution (Rating Based):")
print("-" * 50)
print(df['rating_sentiment'].value_counts())

Sentiment Distribution (VADER):
--------------------------------------------------
sentiment
Positive    985
Neutral     312
Negative    203
Name: count, dtype: int64

Sentiment Distribution (Rating Based):
--------------------------------------------------
rating_sentiment
Positive    1081
Negative     352
Neutral       67
Name: count, dtype: int64


In [36]:
df.to_csv(r'C:\Users\Dell\Data_Science_Project\05_NLP_Sentiments_Project\Data\processed/processed_reviews.csv', index=False)

print("Processed data saved successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

Processed data saved successfully!
Shape: (1500, 9)

Columns: ['username', 'review', 'rating', 'date', 'app', 'cleaned_review', 'processed_review', 'sentiment', 'rating_sentiment']


## 7. Summary & Key Observations

- Total reviews collected: **1459**
- Apps covered: **Zomato, Swiggy, Blinkit**
- Data source: **Google Play Store (Real Data)**
- Positive reviews: **67.5%**
- Negative reviews: **13.9%**
- Neutral reviews: **18.6%**

**Key Finding:** Customers are stricter with ratings than 
with written reviews — rating sentiment shows more negative 
cases than VADER sentiment analysis.

